In [0]:
bounds = spark.sql("""
    SELECT 
        MIN(transaction_date) AS min_date,
        MAX(transaction_date) AS max_date
    FROM silver_transactions
""").collect()[0]

min_date = bounds["min_date"]
max_date = bounds["max_date"]

print(f"Date range: {min_date} → {max_date}")

Date range: 2023-01-01 → 2026-08-18


In [0]:
from pyspark.sql.functions import explode, sequence, lit, to_date

# Build sequence from min to max, then explode into rows
df = spark.range(1).select(
    explode(
        sequence(
            lit(min_date),
            lit(max_date)
        )
    ).alias("date")
)

print(f" {df.count()} dates generated")
display(df.limit(3))

 1326 dates generated


date
2023-01-01
2023-01-02
2023-01-03


In [0]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, quarter, dayofweek, date_format, weekofyear
)

df = (df
    .withColumn("date_key",   date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year",       year(col("date")))
    .withColumn("quarter",    quarter(col("date")))
    .withColumn("month",      month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("week",       weekofyear(col("date")))
    .withColumn("day",        dayofmonth(col("date")))
    .withColumn("weekday",    date_format(col("date"), "EEEE"))
    .withColumn("is_weekend", dayofweek(col("date")).isin([1, 7]).cast("int"))
)

display(df.limit(5))

date,date_key,year,quarter,month,month_name,week,day,weekday,is_weekend
2023-01-01,20230101,2023,1,1,January,52,1,Sunday,1
2023-01-02,20230102,2023,1,1,January,1,2,Monday,0
2023-01-03,20230103,2023,1,1,January,1,3,Tuesday,0
2023-01-04,20230104,2023,1,1,January,1,4,Wednesday,0
2023-01-05,20230105,2023,1,1,January,1,5,Thursday,0


In [0]:
from pyspark.sql.functions import current_timestamp
df = df.withColumn("_gold_updated_at", current_timestamp())
print(" Audit column added")

 Audit column added


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("dim_date")
print(f" dim_date: {spark.table('dim_date').count():,} rows")

 dim_date: 1,326 rows
